# 02 - EDA: findings F1-F10, observations O1-O3 

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from energymind import eda, eda_features, eda_plots as P, io
from energymind.config import load_settings

warnings.filterwarnings("ignore", category=FutureWarning)
%config InlineBackend.figure_format = "png"
plt.rcParams["figure.dpi"] = 110          # notebook display only; files are saved at 300 dpi

settings = load_settings()
df, fx, raw, labels, splits = eda.prepare(settings)   # clean baseline, +EDA features, raw subset, provided labels
findings = eda.compute_findings(df, raw)
observations = eda.compute_observations(fx, splits)
FIG = io.resolve("results/figures")
by_code = {f.code: f for f in [*findings, *observations]}

def metrics(code):
    f = by_code[code]
    print(f"{f.code} - {f.title}")
    for label, value in f.metrics:
        print(f"   {label}: {value}")

print(f"baseline: {df.shape[0]:,} rows x {df.shape[1]} columns | 13 households | figures -> {FIG}")

## Findings at a glance (computed)

In [ ]:
pd.DataFrame(
    [(f.code, f.title, f.consequence) for f in [*findings, *observations]],
    columns=["#", "Finding", "Consequence"],
).style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"})

## A. Dataset overview and data quality

**Fig. 1** is the one-page summary of the panel: record status, temporal coverage, load distribution,
seasonal energy, daily profile and the load-temperature relation (panel f already shows the U-shape that
section D explains). **Fig. 2** shows where the provided anomalies and the 8 missing records sit; isolated
single slots are imputed with the electrical identities rebuilt, the 2 two-slot runs stay NaN.

In [ ]:
fig = P.fig_dataset_overview(fx, FIG); plt.show()

In [ ]:
fig = P.fig_missing_record_map(fx, FIG); plt.show()

## B. Physics consistency and the provided labels (F1, F2, F4)

The generator satisfies the electrical identities almost exactly (F1), so a physics check is an *exact*
detector, and the provided `spike`/`dropout` labels break it by orders of magnitude (F2) - which is why they
are only a sanity-test set. Voltage and frequency sit inside the plausibility bounds (F4, 230 V / 50 Hz).

In [ ]:
fig = P.fig_power_identity(raw, FIG); plt.show()
metrics("F1"); metrics("F2")

In [ ]:
fig = P.fig_physics_residuals(fx, settings.data.physics_residual_tolerance, FIG); plt.show()

In [ ]:
b = settings.data.bounds
fig = P.fig_electrical_parameters(fx, {"voltage_v": b.voltage_v, "frequency_hz": b.frequency_hz}, FIG); plt.show()
metrics("F4")

In [ ]:
fig = P.fig_provided_labels(fx, labels, FIG); plt.show()

## C. Leakage in the provided derived columns (F3)

`power_2h_mean` equals a trailing mean that **includes** the current sample (median error 0), so it moves *at*
the anomaly and would leak the label; it would also not move when we inject. We drop all 64 derived columns and
recompute past-only features (section J).

In [ ]:
fig = P.fig_leakage(raw, FIG); plt.show()
metrics("F3")

## D. Load behaviour: schedule, calendar, temperature (F6, O1)

> **Correction to the first version.** F6 used to say the generator "does not model AC physics" because
> `corr(sub2_cooling, temperature) = 0`. The scatter in Fig. 10(d) shows why that conclusion was wrong: both
> thermal sub-meters grow **symmetrically with the distance from about 20 C** (a V-shape), which a linear
> correlation cannot see. Temperature *is* informative through `|T - 20 C|`; but "heating" and "cooling" are
> both high in winter **and** summer, so those labels are not physical.
>
> The weekly heat-map also shows a small **weekend uplift** rather than "no weekday effect".

In [ ]:
fig = P.fig_weekly_profile(fx, FIG); plt.show()

In [ ]:
fig = P.fig_daily_profiles(fx, FIG); plt.show()
metrics("O1")

In [ ]:
fig = P.fig_monthly_load_temperature(fx, FIG); plt.show()
metrics("F6")

## E. Heterogeneity and distributed resources (O2)

PV follows cloud cover and is zero at night, EV charging is confined to 18:00-08:00 and dominates the
household's power, and the battery charges at midday and discharges in the evening. Consequences for the
detectors: a single global threshold is wrong (EV households are bimodal) and a PV household legitimately
drops far below its peers at midday.

In [ ]:
fig = P.fig_der_profiles(fx, FIG); plt.show()
metrics("O2")

In [ ]:
fig = P.fig_power_distribution(fx, FIG); plt.show()

## F. Weather (F5)

Temperature differs between households at the same instant and the weather *condition* is independent of
temperature and season (80 % of `snow` rows are above 10 C). Weather labels are therefore weak evidence; coherent
events must be injected rather than inferred from the weather columns.

In [ ]:
fig = P.fig_weather(fx, FIG); plt.show()
metrics("F5")

## G. Energy balance and metadata (F7, F8)

Sub-meters are not a closed balance (an unmetered remainder dominates). The `sub-meters > total` rows occur
**only in PV households**, where net metering makes the total smaller than the gross sub-meter sum - so that
check is a PV artefact, not a fault indicator. Metadata contradicts itself, so size/type are not used as
features or explanations.

In [ ]:
fig = P.fig_submeter_balance(fx, FIG); plt.show()
metrics("F7")

In [ ]:
fig = P.fig_metadata(fx, FIG); plt.show()
metrics("F8")

## H. Flat runs and cross-household coherence (F9, F10)

Natural flat runs reach 16 intervals but only at near-zero load, so stuck-at faults at normal load are
detectable by flatness. Households are weakly correlated at 30 minutes and strongly at one day; the 30-minute
matrix shows two clusters, which is what a coherence feature can exploit to separate a genuine network event from a
single-meter fault.

In [ ]:
fig = P.fig_flat_runs(fx, FIG); plt.show()
metrics("F9")

In [ ]:
fig = P.fig_cross_household(fx, FIG); plt.show()
metrics("F10")

## I. Temporal splits and distribution shift 

Seasonality moves the level between splits (validation is the trough, test is rising). Any threshold or baseline
fitted on January-August will be biased out of sample, so thresholds are fixed on validation only and the Step 6
baseline needs a seasonal term (`|T - 20 C|` or a trailing level).

In [ ]:
fig = P.fig_temporal_splits(fx, splits, FIG); plt.show()
metrics("O3")

## J. Feature engineering prototype (blueprint section 6, input to Step 6)

All features are **past-only** (`shift(1)` before rolling) except those that deliberately use the current value;
everything fitted (hour-of-week median/MAD) is fitted on the **training window only**. `features.py` stays the
production implementation; this prototype is what the EDA needs to see the data the way the detectors will.

In [ ]:
pd.DataFrame(eda_features.FEATURE_TABLE, columns=["feature", "definition", "type"]).style.hide(axis="index")

In [ ]:
fig = P.fig_engineered_features(fx, splits, FIG); plt.show()

**Past-only test.** Perturb one *test-period* value and recompute: only the explicit current-value features may
change at that timestamp (the fitted baseline is unaffected because the point is outside the training window).

In [ ]:
hh = "HH_00095_AUS"
cand = df.index[(df["household_id"] == hh) & (df["ts_utc"] >= pd.Timestamp("2020-11-10", tz="UTC"))]
t_idx = cand[100]
df2 = df.copy()
df2.loc[t_idx, "active_power_kw"] *= 5
fx2 = eda_features.engineer_features(df2, splits["train"][1])

feat = [c for c in fx.columns if c not in df.columns and pd.api.types.is_numeric_dtype(fx[c])]
key = (df.loc[t_idx, "household_id"], df.loc[t_idx, "ts_utc"])
a = fx.set_index(["household_id", "ts_utc"])[feat].loc[key].astype(float)
b = fx2.set_index(["household_id", "ts_utc"])[feat].loc[key].astype(float)
changed = sorted(c for c in feat if not np.isclose(a[c], b[c], equal_nan=True))
print("features that changed at t:", changed)
assert set(changed) <= set(eda_features.CURRENT_VALUE), "a past-only feature leaked the current value"
print("OK - every changed feature is a declared current-value feature")